In [ ]:
import os
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('bmh')

PYTHON_PATH = '/home/andreas/miniforge3/envs/symdel/bin/python'
WORKER_SCRIPT = 'symscan_ncpu_worker.py'
SEQ_FILE = '../data/emerson_rep1_sequences.txt'
N_SEQUENCE = 1_000_000
MAX_DISTANCE = 1
DISTANCE_TYPE = 'levenshtein'
N_REPS = 1
MAX_NCPU = os.cpu_count()
#p_core_list = '0,2,4,6,8,10,12,14'

In [ ]:
%%writefile symscan_ncpu_worker.py
import sys
import time

import symscan


def main():
    seq_file, n_sequence, max_distance, distance_type = sys.argv[1:5]
    n_sequence = int(n_sequence)
    max_distance = int(max_distance)

    with open(seq_file) as f:
        seqs = [next(f).strip() for _ in range(n_sequence)]

    t0 = time.perf_counter()
    symscan.get_neighbors_within(seqs, max_distance=max_distance, distance_type=distance_type)
    print(time.perf_counter() - t0)


if __name__ == '__main__':
    main()

In [ ]:
def measure_runtime_seconds(n_cpu, n_sequence=N_SEQUENCE, max_distance=MAX_DISTANCE, distance_type=DISTANCE_TYPE):
    cmd = [PYTHON_PATH, WORKER_SCRIPT, SEQ_FILE, str(n_sequence), str(max_distance), distance_type]
    env = os.environ | {'RAYON_NUM_THREADS': str(n_cpu)}
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return float(result.stdout)

ncpus = np.arange(1, MAX_NCPU + 1)
ncpus

In [ ]:
rows = []
for n_cpu in ncpus:
    for rep in range(N_REPS):
        runtime_s = measure_runtime_seconds(n_cpu)
        rows.append({'algorithm': 'symscan', 'n_cpu': int(n_cpu), 'n_sequence': N_SEQUENCE,
                      'distance': MAX_DISTANCE, 'measure': DISTANCE_TYPE,
                      'runtime_s': runtime_s})
        print(n_cpu, rep, runtime_s)

ncpu_df = pd.DataFrame(rows)
ncpu_df.to_csv('../data/symscan_ncpu_benchmark.csv')
ncpu_df.groupby('n_cpu')['runtime_s'].agg(['mean', 'size'])

In [ ]:
mean = ncpu_df.groupby('n_cpu')['runtime_s'].mean()
x, y = mean.index.values, mean.values
speedup = y[0] / y

fig, ax = plt.subplots(figsize=(3.4, 2.4))
ax.plot(x, speedup, 'o-', label='SymScan')
ax.plot(x, x, '--', color='gray', label='Ideal')
ax.set_xlabel('# CPUs')
ax.set_ylabel('Speedup')
ax.legend()
fig.tight_layout(pad=0.0)
fig.savefig('symscan_ncpu_benchmark.svg')